# 01_ssb_metadata_oppsett – Bygg ssb_metadata

Henter rik metadata fra SSB PXWeb v2 API og lagrer til Delta-tabellen `ssb_metadata`.
Kjøres én gang ved oppsett, og ved behov når du vil oppdatere metadata.

## Relasjon til andre tabeller

| Tabell | Formål | Opprettes av |
|---|---|---|
| `ssb_metadata` | Rik maskinlesbar metadata (tittel, dimensjoner, enhet, KLASS-URN, rå JSON) | Denne notebooken |
| `ssb_config` | Driftstabell (frekvens, lookback, priority, last_loaded_timestamp) | `02_ssb_config_admin` |

De to tabellene er koblet via `tabellnr` = `table_id` (samme SSB-tabellnummer).

## Bruk
1. Sett `TABELLNR_LIST_MANUAL` med tabellnumrene du vil ha
2. Kjør Run All
3. `ssb_metadata` overskrives med oppdatert metadata

Tabellen kan også settes via Fabric pipeline-parameter `TABELLNR_LIST` eller miljøvariabel.

## Bibliotek-modus
Kan lastes som ren funksjonsbibliotek fra en annen notebook via
`%run 01_ssb_metadata_oppsett {"LIBRARY_MODE": true}` – da hentes/skrives
ingenting, kun `fetch_table_metadata()`, `parse_cfg_row()` og
`upsert_metadata_row()` gjøres tilgjengelig. Brukes av `02_ssb_config_admin`.

---

In [ ]:
# ============================================================================
# 1) IMPORTS OG PARAMETERE
# ============================================================================
# Setter opp verktøyene notebooken trenger, og bestemmer HVILKE tabeller det
# skal hentes metadata for i denne kjøringen – enten fra listen lenger ned,
# eller fra en parameter/miljøvariabel satt av en pipeline.

import json
import math
import os
import re
import time
from typing import Any, Dict, List, Optional, Tuple

import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("SSBMetadataOppsett").getOrCreate()

import mssparkutils  # type: ignore

# ----------------------------------------------------------------
# LIBRARY_MODE = True: kun funksjoner/schema lastes (kalt via %run fra
# andre notebooks, f.eks. 02_ssb_config_admin). Ingen henting/skriving skjer.
# ----------------------------------------------------------------
LIBRARY_MODE = False


# ----------------------------------------------------------------
# Tabellnummer: sett manuelt her (vinner over pipeline-parameter)
# ----------------------------------------------------------------
TABELLNR_LIST_MANUAL: List[str] = [
    # Legg til dine tabellnumre her:
    # "07459",  # Befolkning etter region, kjønn og alder
    # "12880",  # Befolkningsendringer
]


def _get_job_parameter(name: str) -> Optional[str]:
    try:
        return mssparkutils.env.getJobParameter(name)
    except Exception:
        return None


def _resolve_tabellnr_list() -> Tuple[List[str], str]:
    """Prioritert oppløsning: manuell liste → pipeline-parameter → miljøvariabel"""
    if TABELLNR_LIST_MANUAL:
        ids = [t.strip() for t in TABELLNR_LIST_MANUAL if t.strip()]
        return ids, "manual"
    raw = _get_job_parameter("TABELLNR_LIST")
    if raw:
        ids = [t.strip() for chunk in raw.splitlines() for t in chunk.split(",") if t.strip()]
        return ids, "job-parameter"
    raw = os.getenv("TABELLNR_LIST", "")
    if raw:
        ids = [t.strip() for chunk in raw.splitlines() for t in chunk.split(",") if t.strip()]
        return ids, "env"
    return [], "empty"


# API-innstillinger
PXWEB_BASE  = os.getenv("PXWEB_BASE", "https://data.ssb.no/api/pxwebapi/v2")
LANG        = "no"
STOP_ON_ERROR = False   # False = behandle alle tabeller selv om én feiler
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "4"))
BACKOFF_BASE = float(os.getenv("BACKOFF_BASE", "1.5"))

if not LIBRARY_MODE:
    TABELLNR_LIST, TABELLNR_SOURCE = _resolve_tabellnr_list()

    print("==== 01_ssb_metadata_oppsett – Bygger ssb_metadata ====")
    print(f"PXWEB_BASE       = {PXWEB_BASE}")
    print(f"LANG             = {LANG}")
    print(f"TABELLNR_SOURCE  = {TABELLNR_SOURCE}")
    print(f"TABELLER (n={len(TABELLNR_LIST)}) = {TABELLNR_LIST}")
    assert len(TABELLNR_LIST) > 0, (
        "Ingen tabellnr funnet. Fyll TABELLNR_LIST_MANUAL eller sett TABELLNR_LIST "
        "som pipeline-parameter / miljøvariabel."
    )
else:
    print("==== 01_ssb_metadata_oppsett – lastet som bibliotek (LIBRARY_MODE) ====")


In [ ]:
# ============================================================================
# 2) HJELPEFUNKSJONER (HTTP + parsing)
# ============================================================================
# Funksjoner for å snakke med SSB sitt API. fetch_table_metadata prøver flere
# ulike URL-varianter til den finner en som svarer, siden SSB ikke alltid er
# konsekvent her. _infer_frequency_from_values gjetter oppdateringsfrekvens
# (daglig/ukentlig/månedlig/kvartalsvis/årlig) ut fra mønsteret i tidsseriens
# verdier, for de tilfellene SSB ikke oppgir frekvensen eksplisitt.

def _get_json(url: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """GET med norsk Accept-Language og eksponentiell retry/backoff."""
    last_err: Exception = RuntimeError("Ingen forsøk gjort")
    headers = {"Accept": "application/json", "Accept-Language": "no"}
    for i in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            if 200 <= r.status_code < 300:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"Temporary HTTP {r.status_code}: {r.text[:200]}")
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            last_err = e
            time.sleep(BACKOFF_BASE ** i + 0.05 * i)
    raise RuntimeError(f"GET feilet etter {MAX_RETRIES} forsøk: {last_err}")


def fetch_table_metadata(table_id: str) -> Dict[str, Any]:
    """
    Henter metadata for en SSB-tabell på norsk.
    Prøver /no/-sti først, deretter fallback med ?lang=no.
    """
    candidates = [
        (f"{PXWEB_BASE}/no/tables/{table_id}/data", {"content": "metadata"}),
        (f"{PXWEB_BASE}/tables/{table_id}/data",    {"lang": "no", "content": "metadata"}),
        (f"{PXWEB_BASE}/no/tables/{table_id}",      {}),
        (f"{PXWEB_BASE}/tables/{table_id}",         {"lang": "no"}),
    ]
    errors: List[str] = []
    for url, params in candidates:
        try:
            data = _get_json(url, params)
            # Normaliser label hvis det er et språk-mappet objekt
            if isinstance(data.get("label"), dict):
                data["label"] = (
                    data["label"].get("no")
                    or data["label"].get("nb")
                    or data["label"].get("nn")
                    or next(iter(data["label"].values()), "")
                )
            return data
        except Exception as e:
            errors.append(f"{url} → {e}")
    raise RuntimeError(
        f"Kunne ikke hente metadata for {table_id}. Forsøkt: " + " | ".join(errors)
    )


def _pick_no(x: Any) -> str:
    """Velg norsk verdi fra flerspråklig dict, eller returner x som string."""
    if isinstance(x, dict):
        return (
            x.get("no") or x.get("nb") or x.get("nn")
            or next(iter(x.values()), "")
        ) or ""
    return x or ""


def _extract_time_values(meta: Dict[str, Any]) -> List[str]:
    """Hent alle tidsverdier fra metadata (støtter ulike PXWeb-strukturer)."""
    dim = meta.get("dimension", {})
    if not isinstance(dim, dict):
        return []
    tid_key = next((k for k in dim if k.lower() in ("tid", "time")), None)
    if not tid_key:
        return []
    tid_dim = dim.get(tid_key, {})
    if not isinstance(tid_dim, dict):
        return []
    cat = tid_dim.get("category", {})
    labels = cat.get("label", {})
    if isinstance(labels, dict) and labels:
        return list(labels.values())
    idx = cat.get("index", {})
    if isinstance(idx, dict) and idx:
        return list(idx.keys())
    return tid_dim.get("values") or tid_dim.get("value") or []


def _infer_frequency_from_values(time_values: List[str]) -> Optional[str]:
    """Utled oppdateringsfrekvens ved å matche tidsverdier mot kjente mønstre."""
    if not time_values:
        return None
    samples = time_values[-10:]
    patterns = [
        ("daglig",      re.compile(
            r"^\d{4}[-/](0[1-9]|1[0-2])[-/](0[1-9]|[12]\d|3[01])$"
            r"|^(0?[1-9]|[12]\d|3[01])\.(0?[1-9]|1[0-2])\.\d{4}$"
        )),
        ("ukentlig",    re.compile(
            r"^\d{4}\s*[WwUu]\s?(0[1-9]|[1-4]\d|5[0-3])$"
            r"|^uke\s+\d{1,2}\s+\d{4}$", re.IGNORECASE
        )),
        ("månedlig",    re.compile(
            r"^\d{4}\s*[Mm]\s?(0[1-9]|1[0-2])$"
            r"|^\d{4}[-/](0[1-9]|1[0-2])$"
            r"|^(jan|feb|mar|apr|mai|jun|jul|aug|sep|okt|nov|des)\s+\d{4}$", re.IGNORECASE
        )),
        ("kvartalsvis", re.compile(
            r"^\d{4}\s*[KQ]\s?[1-4]$"
            r"|^\d{4}[-/]\s*[Qq][1-4]$"
        )),
        ("årlig",       re.compile(r"^\s*\d{4}\s*$")),
    ]
    counts = {label: sum(bool(p.match(v)) for v in samples) for label, p in patterns}
    best = max(counts, key=counts.get)
    return best if counts[best] > 0 else None


print("✅ Hjelpefunksjoner definert")

In [ ]:
# ============================================================================
# 3) PARSING: bygg én rad for ssb_metadata
# ============================================================================
# Den rå responsen fra SSB er teknisk og ikke alltid strukturert likt fra
# tabell til tabell. Denne funksjonen plukker ut det vi faktisk bryr oss om
# – tittel, dimensjoner, måleenhet, kontaktperson, klassifikasjonskoder
# (KLASS) og lignende – og samler det i én ryddig, flat rad.

def parse_cfg_row(table_id: str, meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Ekstraherer alle relevante felt fra PXWeb metadata-respons
    og returnerer en flat dict som matcher ssb_metadata-schema.
    """
    # Basisfelt
    title   = (
        _pick_no(meta.get("label"))
        or _pick_no(meta.get("title"))
        or meta.get("tableLabel", "")
        or meta.get("description", "")
    )
    source  = meta.get("source") or meta.get("sourceLabel") or ""
    updated = (
        meta.get("updated")
        or meta.get("lastUpdated")
        or meta.get("updateTime")
        or ""
    )

    # Dimensjoner
    if isinstance(meta.get("id"), list):
        dims = meta["id"]
    elif isinstance(meta.get("dimension"), dict):
        dims = list(meta["dimension"].keys())
    else:
        dims = []

    role    = meta.get("role") or {}
    metrics = role.get("metric", []) if isinstance(role, dict) else []
    has_time = any(d.lower() in ("tid", "time") for d in dims)
    has_geo  = any(d.lower() in ("region", "geo") for d in dims)

    # Innholdskode (ContentsCode) – metrikkbeskrivelse og enhet
    metric_label = unit_base = None
    unit_decimals: Optional[int] = None
    try:
        cc = meta["dimension"]["ContentsCode"]["category"]
        if isinstance(cc.get("label"), dict) and cc["label"]:
            metric_label = list(cc["label"].values())[0]
        if isinstance(cc.get("unit"), dict) and cc["unit"]:
            first_key = next(iter(cc["unit"]))
            u = cc["unit"][first_key]
            unit_base     = u.get("base")
            unit_decimals = u.get("decimals")
    except Exception:
        pass

    # extension.px
    ext = meta.get("extension") if isinstance(meta.get("extension"), dict) else {}
    px  = ext.get("px")        if isinstance(ext.get("px"), dict)        else {}

    subject_area        = px.get("subject-area")
    subject_code        = px.get("subject-code")
    matrix              = px.get("matrix")
    official_statistics = px.get("official-statistics")
    aggregallowed       = px.get("aggregallowed")
    language_code       = px.get("language")
    heading = px.get("heading") if isinstance(px.get("heading"), list) else []
    stub    = px.get("stub")    if isinstance(px.get("stub"),    list) else []

    # Kontaktinfo
    contact_name = contact_email = contact_phone = None
    contacts = ext.get("contact")
    if isinstance(contacts, list) and contacts:
        c0 = contacts[0]
        if isinstance(c0, dict):
            contact_name  = c0.get("name")
            contact_phone = c0.get("phone")
            contact_email = c0.get("mail") or c0.get("email")

    # KLASS-URNer per dimensjon
    klass_urns: Dict[str, List[str]] = {}
    dim_obj = meta.get("dimension", {})
    if isinstance(dim_obj, dict):
        for dk, dobj in dim_obj.items():
            try:
                links = dobj.get("link", {}).get("describedby", [])
                urns: List[str] = []
                for lnk in links:
                    extn = lnk.get("extension", {})
                    if isinstance(extn, dict):
                        v = extn.get(dk)
                        if isinstance(v, str):
                            urns.extend(v.split())
                if urns:
                    klass_urns[dk] = urns
            except Exception:
                continue

    # Tidsinfo og frekvens
    time_vals  = _extract_time_values(meta) if has_time else []
    time_first = time_vals[0]  if time_vals else None
    time_last  = time_vals[-1] if time_vals else None
    time_count = len(time_vals) if time_vals else None

    raw_freq = (
        meta.get("updateFrequency")
        or meta.get("updateInterval")
        or meta.get("frequency")
    )
    explicit_freq = _pick_no(raw_freq) if isinstance(raw_freq, dict) else (raw_freq or None)

    inferred_freq: Optional[str] = None
    if not explicit_freq:
        inferred_freq = _infer_frequency_from_values(time_vals)
        if not inferred_freq:
            # Fallback: prøv label på Tid-dimensjonen
            try:
                tkey = next((k for k in dim_obj if k.lower() in ("tid", "time")), None)
                if tkey:
                    tl = (_pick_no(dim_obj[tkey].get("label")) or "").lower()
                    if   "år"      in tl: inferred_freq = "årlig"
                    elif "kvartal" in tl: inferred_freq = "kvartalsvis"
                    elif "måned"   in tl: inferred_freq = "månedlig"
                    elif "uke"     in tl: inferred_freq = "ukentlig"
                    elif "dag"     in tl: inferred_freq = "daglig"
            except Exception:
                pass

    return {
        "tabellnr":                  table_id,
        "lang":                      LANG,
        "title":                     title,
        "source":                    source,
        "updated_utc":               updated,
        "has_time_dim":              has_time,
        "has_geo_dim":               has_geo,
        "dims":                      dims,
        "metrics":                   metrics,
        "metric_label":              metric_label,
        "unit_base":                 unit_base,
        "unit_decimals":             unit_decimals,
        "subject_area":              subject_area,
        "subject_code":              subject_code,
        "matrix":                    matrix,
        "official_statistics":       official_statistics,
        "aggregallowed":             aggregallowed,
        "language_code":             language_code,
        "heading":                   heading,
        "stub":                      stub,
        "contact_name":              contact_name,
        "contact_email":             contact_email,
        "contact_phone":             contact_phone,
        "update_frequency":          explicit_freq,
        "update_frequency_inferred": inferred_freq,
        "time_first":                time_first,
        "time_last":                 time_last,
        "time_count":                time_count,
        "klass_urns":                klass_urns,
        "pxweb_base":                PXWEB_BASE,
        "raw_metadata_json":         json.dumps(meta, ensure_ascii=False),
    }


print("✅ parse_cfg_row definert")

In [ ]:
# ============================================================================
# 4) HENT METADATA FOR ALLE TABELLER
# ============================================================================
# Går gjennom tabellnumrene ett og ett og henter metadata for hver. Hopper
# ikke ut av løkken om én tabell feiler (med mindre STOP_ON_ERROR er satt) –
# feilene samles opp og vises i sammendraget til slutt, slik at én dårlig
# tabell ikke stopper resten. Kjøres kun når notebooken IKKE er lastet som
# bibliotek (se LIBRARY_MODE i celle 1).

if not LIBRARY_MODE:
    rows:   List[Dict[str, Any]] = []
    errors: List[Dict[str, str]] = []

    print(f"\n🚀 Henter metadata for {len(TABELLNR_LIST)} tabeller...\n")

    for i, t in enumerate(TABELLNR_LIST, 1):
        print(f"[{i}/{len(TABELLNR_LIST)}] {t}", end="  ")
        try:
            meta = fetch_table_metadata(t)
            row  = parse_cfg_row(t, meta)
            rows.append(row)
            freq_str = row["update_frequency"] or row["update_frequency_inferred"] or "ukjent"
            print(
                f"OK  | {row['title'][:60]:<60} "
                f"| metric='{row['metric_label']}' "
                f"| unit='{row['unit_base']}' "
                f"| freq={freq_str}"
            )
        except Exception as e:
            msg = str(e)
            errors.append({"tabellnr": t, "error": msg})
            print(f"FEIL | {msg}")
            if STOP_ON_ERROR:
                raise
        time.sleep(0.3)  # høflig pause mot API

    print(f"\n📊 Ferdig: {len(rows)} OK, {len(errors)} feil")


In [ ]:
# ============================================================================
# 5) SCHEMA OG UPSERT AV ssb_metadata
# ============================================================================
# Definerer den faste strukturen på katalogtabellen ssb_metadata, og lagrer
# radene som ble hentet i forrige celle. upsert_metadata_row oppdaterer en
# eksisterende rad hvis tabellen allerede finnes i katalogen, ellers legges
# den til – slik kan man kjøre dette for én og én tabell (f.eks. fra
# 02_ssb_config_admin) uten å overskrive resten av katalogen.

CFG_SCHEMA = T.StructType([
    T.StructField("tabellnr",                  T.StringType(),                         False),
    T.StructField("lang",                      T.StringType(),                         False),
    T.StructField("title",                     T.StringType(),                         True),
    T.StructField("source",                    T.StringType(),                         True),
    T.StructField("updated_utc",               T.StringType(),                         True),
    T.StructField("has_time_dim",              T.BooleanType(),                        True),
    T.StructField("has_geo_dim",               T.BooleanType(),                        True),
    T.StructField("dims",                      T.ArrayType(T.StringType()),             True),
    T.StructField("metrics",                   T.ArrayType(T.StringType()),             True),
    T.StructField("metric_label",              T.StringType(),                         True),
    T.StructField("unit_base",                 T.StringType(),                         True),
    T.StructField("unit_decimals",             T.IntegerType(),                        True),
    T.StructField("subject_area",              T.StringType(),                         True),
    T.StructField("subject_code",              T.StringType(),                         True),
    T.StructField("matrix",                    T.StringType(),                         True),
    T.StructField("official_statistics",       T.BooleanType(),                        True),
    T.StructField("aggregallowed",             T.BooleanType(),                        True),
    T.StructField("language_code",             T.StringType(),                         True),
    T.StructField("heading",                   T.ArrayType(T.StringType()),             True),
    T.StructField("stub",                      T.ArrayType(T.StringType()),             True),
    T.StructField("contact_name",              T.StringType(),                         True),
    T.StructField("contact_email",             T.StringType(),                         True),
    T.StructField("contact_phone",             T.StringType(),                         True),
    T.StructField("update_frequency",          T.StringType(),                         True),
    T.StructField("update_frequency_inferred", T.StringType(),                         True),
    T.StructField("time_first",                T.StringType(),                         True),
    T.StructField("time_last",                 T.StringType(),                         True),
    T.StructField("time_count",                T.IntegerType(),                        True),
    T.StructField("klass_urns",                T.MapType(T.StringType(),
                                                         T.ArrayType(T.StringType())), True),
    T.StructField("pxweb_base",                T.StringType(),                         True),
    T.StructField("raw_metadata_json",         T.StringType(),                         True),
])


def upsert_metadata_row(row: Dict[str, Any]) -> None:
    """
    Sett inn eller oppdater én rad i statbank_staging.pipeline.ssb_metadata (MERGE på tabellnr).
    Kan kalles for én tabell om gangen – f.eks. fra 02_ssb_config_admin når en
    tabell legges til/resettes – uten å påvirke resten av katalogen.
    """
    from delta.tables import DeltaTable

    new_df = spark.createDataFrame([row], schema=CFG_SCHEMA)

    if spark.catalog.tableExists("statbank_staging.pipeline.ssb_metadata"):
        (
            DeltaTable.forName(spark, "statbank_staging.pipeline.ssb_metadata")
            .alias("target")
            .merge(new_df.alias("source"), "target.tabellnr = source.tabellnr")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        new_df.write.format("delta").mode("overwrite").saveAsTable("statbank_staging.pipeline.ssb_metadata")

    print(f"   ssb_metadata oppdatert: {row['tabellnr']}")


if not LIBRARY_MODE:
    if rows:
        for row in rows:
            upsert_metadata_row(row)
        print(f"✅ ssb_metadata oppdatert ({len(rows)} tabeller)")
    else:
        print("⚠️  Ingen rader å skrive – ssb_metadata ikke oppdatert")


In [ ]:
# ============================================================================
# 6) VISNING OG FEILOPPSUMMERING
# ============================================================================
# Viser et sammendrag av kjøringen: en tabell med metadataen som ble hentet,
# og – hvis noe gikk galt – en liste over hvilke tabeller som feilet og hvorfor.

if not LIBRARY_MODE:
    if rows:
        print("\n==== ssb_metadata – standardvisning ====")
        display(
            spark.table("statbank_staging.pipeline.ssb_metadata")
            .select(
                "tabellnr", "title", "subject_area",
                "updated_utc",
                "metric_label", "unit_base",
                "subject_code",
                "update_frequency", "update_frequency_inferred",
                "contact_name", "contact_email",
                "official_statistics", "aggregallowed",
                "has_time_dim", "has_geo_dim",
            )
            .orderBy(F.col("tabellnr"))
        )

    if errors:
        print("\n==== Feiloppsummering ====")
        for e in errors:
            print(f"  {e['tabellnr']}: {e['error']}")

    print(f"\n{'='*60}")
    print(f"✅ 01_ssb_metadata_oppsett ferdig: {len(rows)} tabeller i ssb_metadata")
    if errors:
        print(f"⚠️  {len(errors)} tabeller feilet (se feiloppsummering over)")
    print(f"{'='*60}")
